# AST Code-Edit — GRPO Training on a Colab T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nithin062006/scaler/blob/main/training/notebook.ipynb)

**Pipeline:**
1. Clone repo + install deps
2. Explore the environment — AST parsing → DAG → reward demo
3. Baseline eval (untrained Qwen2.5-0.5B)
4. GRPO training with LoRA
5. Trained eval + plots

**Approach:** The agent sees a Python module as a DAG (AST-extracted nodes: module / function / class / import; edges: contains / calls / imports). Its task is to fill in a STUB function body. Reward = compile check + test-case pass rate. GRPO trains the model to improve this reward via group-relative policy optimization.

Expected wall-clock on T4: ~15–30 min.

## 1. Setup

In [ ]:
import os, subprocess, pathlib

REPO_URL = 'https://github.com/nithin062006/scaler.git'
cwd = pathlib.Path(os.getcwd())

if (cwd / 'env').exists() and (cwd / 'training').exists():
    print(f'Already inside repo: {cwd}')
elif (cwd / 'scaler').exists():
    os.chdir('scaler')
    print(f'Cd-ed: {os.getcwd()}')
else:
    subprocess.check_call(['git', 'clone', '-q', REPO_URL, 'scaler'])
    os.chdir('scaler')
    print(f'Cloned: {os.getcwd()}')

print(os.listdir('.'))

In [ ]:
# Install core deps
%pip install -q -e ".[training]"
# Optional: Unsloth for 2-4x faster training (uncomment if available on your Colab)
# %pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 2. Explore the environment

In [ ]:
import sys
sys.path.insert(0, '.')

from env.environment import ASTCodeEditEnvironment
from env.models import CodeEditAction
from env.tasks import TASK_BANK

# Show all tasks
print('Available tasks:')
for tid, task in TASK_BANK.items():
    print(f'  {tid}: {task.description[:60]}...')
print()

In [ ]:
from env.ast_parser import parse_source, dag_to_text

# Parse the palindrome task's source and show the DAG
task = TASK_BANK['t0.palindrome']
dag = parse_source(task.source_code, task.module_name)

print('=== AST → DAG ===')
print(dag_to_text(dag))
print()
print('Stub functions:', dag.stub_functions())
print('Callers of is_palindrome:', dag.callers_of('is_palindrome'))

In [ ]:
# Demo: run one episode manually
env = ASTCodeEditEnvironment()

obs = env.reset(task_id='t0.palindrome')
print('=== Observation after reset ===')
print('Task ID       :', obs.task_id)
print('Target        :', obs.target_function, obs.target_signature)
print('Called by     :', obs.callers)
print('Task desc     :', obs.task_description)
print()

# Good action
obs, reward, done = env.step(CodeEditAction(code='return s == s[::-1]'))
print('=== After correct action ===')
print(f'Reward        : {reward:.2f}')
print(f'Compile OK    : {obs.compile_ok}')
print(f'Tests passed  : {obs.tests_passed} / {obs.tests_total}')
print()

# Bad action
env.reset(task_id='t0.palindrome')
obs, reward, done = env.step(CodeEditAction(code='return BROKEN CODE !!!'))
print('=== After bad action ===')
print(f'Reward        : {reward:.2f}')
print(f'Compile OK    : {obs.compile_ok}')

## 3. Build prompt dataset

In [ ]:
from training.train import build_user_prompt, SYSTEM_PROMPT

# Show what the agent sees for the palindrome task
print('=== Agent prompt (user turn) ===')
print(build_user_prompt('t0.palindrome'))

## 4. Baseline eval (before training)

In [ ]:
from pathlib import Path
from training.config import TrainConfig
from training.train import load_model_and_tokenizer, evaluate

cfg = TrainConfig(
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    n_eval_per_task=4,
    max_completion_length=300,
    temperature=0.9,
    out_dir=Path('outputs'),
    plots_dir=Path('plots'),
)

model, tokenizer = load_model_and_tokenizer(cfg)
baseline = evaluate(model, tokenizer, cfg)

print(f'Baseline mean reward: {baseline["mean"]:.3f}')
for tid, rs in baseline['per_task'].items():
    import statistics
    print(f'  {tid}: {statistics.mean(rs):.3f}')

## 5. GRPO training

In [ ]:
from training.train import run

cfg = TrainConfig(
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    use_lora=True,
    lora_r=16,
    lora_alpha=32,
    num_generations=4,       # G completions per prompt
    epochs=3,
    learning_rate=5e-6,
    batch_size=1,
    gradient_accumulation_steps=4,
    samples_per_task=10,
    n_eval_per_task=4,
    max_completion_length=300,
    temperature=0.9,
    out_dir=Path('outputs'),
    plots_dir=Path('plots'),
)

summary = run(cfg)

print('\n' + '='*60)
print(f'Baseline mean : {summary["baseline"]["mean"]:.3f}')
print(f'Trained  mean : {summary["trained"]["mean"]:.3f}')
print(f'Delta         : {summary["trained"]["mean"] - summary["baseline"]["mean"]:+.3f}')
print('='*60)

## 6. Show plots

In [ ]:
from IPython.display import Image, display

for name in ['comparison.png', 'reward_curve.png', 'loss_curve.png']:
    p = Path('plots') / name
    if p.exists():
        print(f'--- {name} ---')
        display(Image(str(p)))

## 7. Commit artifacts

```bash
git add plots/ outputs/
git commit -m "Training run: GRPO reward improvement"
git push
```